# BiLSTM Adversarial Examples: Test

This notebook generates gradient-guided character edits against the trained BiLSTM, validates them, and evaluates the model on the original test set plus non-duplicate adversarial rows.

In [ ]:
import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

sys.path.insert(0, "../src")
import generate_bilstm_adversarial as attack_impl

## Setup

The model uses the project character vocabulary, a 256-character input limit, and a maximum edit budget of 3.

In [ ]:
SPLIT = "test"
N = None  # set to a small number such as 10 for a smoke test
BATCH_SIZE = 64
TOP_POSITIONS = 3
SEED = 42

CHECKPOINT = Path("../models/bilstm_model1.pt")
OUTPUT = Path("../data/bilstm_adv_test.csv")
PARTIAL = Path("../data/bilstm_adv_test_partial.csv")
RESULTS_DIR = Path("../results")

config, vocab = attack_impl.load_config()
MAX_LEN = config["max_len"]
K = config["budget"]
INDEX_TO_STRING = vocab["index_to_string"]
CHAR_TO_IDX = {character: index for index, character in enumerate(INDEX_TO_STRING)}
PAD_IDX = vocab["pad_idx"]
UNK_IDX = vocab["unk_idx"]
DEVICE = "cuda" if attack_impl.torch.cuda.is_available() else "mps" if attack_impl.torch.backends.mps.is_available() else "cpu"
torch = attack_impl.torch
torch.manual_seed(SEED)
np.random.seed(SEED)
print("device:", DEVICE, "| split:", SPLIT, "| max_len:", MAX_LEN, "| k:", K)

## Load Data and Model

In [ ]:
splits = attack_impl.load_splits()
model = attack_impl.load_model(CHECKPOINT, len(CHAR_TO_IDX), PAD_IDX, DEVICE)
embedding = model.embedding.weight
valid_embedding_ids = torch.tensor([index for character, index in CHAR_TO_IDX.items() if character not in {"<PAD>", "<UNK>"}], dtype=torch.long, device=DEVICE)
p_phish = attack_impl.make_predictor(model, CHAR_TO_IDX, UNK_IDX, PAD_IDX, MAX_LEN, DEVICE)
print("checkpoint:", CHECKPOINT, "| vocabulary:", len(CHAR_TO_IDX))

## Select Attack Targets

Only phishing URLs that the clean BiLSTM currently detects are attacked. The original phishing probability is kept as `p_clean`.

In [ ]:
phishing = splits[SPLIT][splits[SPLIT].label == 0].copy()
p_all = p_phish(phishing.URL.tolist())
targets = phishing[p_all >= 0.5].copy()
targets["p_clean"] = p_all[p_all >= 0.5]
if N is not None:
    targets = targets.sample(min(N, len(targets)), random_state=SEED)
print(f"phishing: {len(phishing)} | missed: {(p_all < 0.5).sum()} | to attack: {len(targets)}")

## Attack

The BiLSTM is character-based, so each gradient position already corresponds to one URL character. The shared attack function ranks positions using gradients through the character embedding matrix, then tries valid substitutions, insertions, deletions, and appends.

In [ ]:
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
PARTIAL.parent.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if PARTIAL.exists():
    completed = pd.read_csv(PARTIAL, keep_default_na=False)
else:
    completed = pd.DataFrame(columns=["orig_URL", "URL"])
todo = targets[~targets.URL.isin(completed.orig_URL)]

start_time = time.time()
for start in tqdm(range(0, len(todo), BATCH_SIZE), desc=f"attack {SPLIT}"):
    chunk = todo.iloc[start:start + BATCH_SIZE]
    adversarial = attack_impl.attack_batch(chunk.URL.tolist(), chunk.p_clean.tolist(), model, embedding, valid_embedding_ids, CHAR_TO_IDX, UNK_IDX, PAD_IDX, MAX_LEN, K, TOP_POSITIONS, DEVICE, p_phish)
    pd.DataFrame({"orig_URL": chunk.URL.values, "URL": adversarial}).to_csv(PARTIAL, mode="a", header=not PARTIAL.exists(), index=False)
print("seconds per URL:", round((time.time() - start_time) / max(1, len(todo)), 4))

## Validate and Save

In [ ]:
attack_map = dict(pd.read_csv(PARTIAL, keep_default_na=False).values) if PARTIAL.exists() else {}
adv_df = pd.DataFrame({"orig_URL": targets.URL.values, "URL": [attack_map.get(url, url) for url in targets.URL], "label": 0, "p_clean": targets.p_clean.values})
adv_df["p_adv"] = p_phish(adv_df.URL.tolist())
adv_df["n_edits"] = [attack_impl.edit_distance(original, edited) for original, edited in zip(adv_df.orig_URL, adv_df.URL)]
adv_df["flipped"] = adv_df.p_adv < 0.5
reasons = pd.Series([attack_impl.invalid_reason(original, edited, K) for original, edited in zip(adv_df.orig_URL, adv_df.URL)])
print(reasons.replace("", "valid").value_counts())
assert (reasons == "").all()
adv_df["duplicate"] = (adv_df.n_edits > 0) & (adv_df.URL.isin(splits[SPLIT].URL) | adv_df.URL.duplicated())
summary = {"model": "bilstm", "split": SPLIT, "phishing": len(phishing), "missed_by_model": int((p_all < 0.5).sum()), **attack_impl.attack_summary(adv_df, K), "n_edits": {int(count): int(number) for count, number in adv_df.n_edits.value_counts().sort_index().items()}}
adv_df.to_csv(OUTPUT, index=False)
if PARTIAL.exists():
    PARTIAL.unlink()
with (RESULTS_DIR / "bilstm_adv_test_summary.json").open("w") as file:
    json.dump(summary, file, indent=1)
display(adv_df.head())
summary

## Evaluate Original Plus Adversarial Test Rows

In [ ]:
edited = attack_impl.adversarial_rows(adv_df)
test = splits["test"]
p_test = p_phish(test.URL.tolist())
both = pd.concat([test[["URL", "label"]], edited[["URL", "label"]]], ignore_index=True)
p_both = np.concatenate([p_test, edited.p_adv.values])
m_orig = attack_impl.evaluate(test.label, p_test)
m_adv = attack_impl.evaluate(both.label, p_both)
metrics = attack_impl.adversarial_metrics(adv_df.label, adv_df.p_clean, adv_df.p_adv)
metrics["adversarial_recall"] = attack_impl.adversarial_recall(edited.p_adv)
metrics.update(attack_impl.deltas(m_orig, m_adv))
attack_impl.save_preds(RESULTS_DIR / "bilstm_test_adversarial.csv", both.URL, both.label, p_both)
with (RESULTS_DIR / "bilstm_test_adversarial.json").open("w") as file:
    json.dump({**m_adv, **metrics}, file, indent=1)
display(pd.DataFrame({"test_original": m_orig, "test_original + test_adversarial": m_adv}).drop("confusion_matrix").T.astype(float).round(4))
metrics